In [1]:
import os
from pathlib import Path
import torch 
import numpy as np

from ml.inference import load_checkpoint

## On charge le modèle MAEv1 après premier run de préentrainement
base = Path.cwd()

ckpt_dir = Path(os.path.join(base, '..', '..', 'outputs', 'ml', 'pretrain', '2026-08-05_13-36-04', 'checkpoints', 'best.pt'))

device = torch.device("cpu")
pretrained_model, cfg, mean, scale = load_checkpoint(ckpt_dir, device)
print(pretrained_model)


TimeSeriesMAE(
  (patch_embedding): PatchEmbedding(
    (proj): Conv1d(11, 64, kernel_size=(12,), stride=(12,))
  )
  (encoder_pos_embedding): LearnedPositionalEmbedding(
    (PE): Embedding(8, 64)
  )
  (decoder_pos_embedding): LearnedPositionalEmbedding(
    (PE): Embedding(9, 32)
  )
  (encoder_blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (MSA): MultiHeadSelfAttention(
        (query_matrix): Linear(in_features=64, out_features=64, bias=True)
        (key_matrix): Linear(in_features=64, out_features=64, bias=True)
        (value_matrix): Linear(in_features=64, out_features=64, bias=True)
        (output_proj): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (MLP): MLP(
        (dense1): Linear(in_features=64, out_features=256, bias=True)
        (d

In [2]:
## on charge les poids de l'encoder du modèle préentrainé sur un backbone avec les mêmes paramètres (cf MAEv1.yaml pour l'archi de l'encoder)
from ml.model import VanillaViT

encoder_sd = pretrained_model.encoder_state_dict() ## donne directement les poids de l'encodeur avec les bons mapping


backbone = VanillaViT(
    n_features = 11, 
    n_epochs= 96, 
    patch_size=12, 
    embed_dim=64,
    n_attn_heads=4,
    n_blocks=4,
    expansion_factor=4,
    dropout_rate=0.1
)
missing, unexpected = backbone.load_state_dict(encoder_sd, strict=False) # les poids sont bien chargés. 



In [3]:
from ml.datahandler import load_spacetrack_objects, build_features, diff_cols_spacetrack

data_dir = Path(os.path.join(base, '..' , '..', 'data', 'raw', 'spacetrack'))
objects = load_spacetrack_objects(data_dir)

per_obj = {}
for key, df in objects.items(): 
    features, feature_cols = build_features(df, spacetrack=True)
    X = (features[feature_cols].to_numpy(np.float32) - mean) / scale
    per_obj[key] = X

backbone.eval()
backbone.to(device)

window_size = 96 
batch_size = 256

out = {}
for norad, X in per_obj.items():
    if len(X) < window_size : 
        continue 
    windows = np.lib.stride_tricks.sliding_window_view(X, window_size, axis=0) # (L,F,W)

    cls_representations = []

    for i in range(0, len(windows), batch_size): 
        x = torch.from_numpy(np.ascontiguousarray(windows[i:i+batch_size])).float().to(device)

        with torch.no_grad(): 
            representation = backbone(x)
        cls_representations.append(representation[:,0,:].cpu().numpy()) ## on ne stocke que le cls token !

    if cls_representations:
        out[norad] = np.concatenate(cls_representations, axis=0).mean(axis=0) 


In [33]:
import hdbscan, dbscan
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## une représentation par objet : token CLS (indice 0) moyenné sur toutes les fenêtres
norad_ids = list(out.keys())
X_obj = np.stack([out[n] for n in norad_ids])
X_scaled = StandardScaler().fit_transform(X_obj)

clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=5)
labels = clusterer.fit_predict(X_scaled)

n_clusters = len(set(labels))
print(len(norad_ids))


14993


In [43]:

import matplotlib.pyplot as plt
import pandas as pd

clusterized_ids = [n for n in norad_ids if n in labels and n in out and labels[n]!=-1]
print(norad_ids)
X_df = pd.DataFrame({n : out[n] for n in clusterized_ids})

print(X_df)
def viz_3d_on_clusters(df):
    """X is a (14 993, 64) ndarray of already scaled vectors, we reduce the dimension from 64 to 3 with principal components analysis, and show the clusters
    """
    pca = PCA(n_components=3)
    X_3d = pca.fit_transform(X)

    fig = plt.figure(figsize= (10,10))
    ax = fig.add_subplot(projection='3d')

    ax.scatter(X_3d[:,0], X_3d[:,1] , X_3d[:,2], c=labels, cmap='tab10', s=5, alpha=0.8)
    ax.view_init(elev=0, azim=110)
    plt.tight_layout()
    plt.show()

def viz_2d_on_clusters(X):
    """X is a (14 993, 64) ndarray of already scaled vectors, we reduce the dimension from 64 to 2 with principal components analysis, and show the clusters
    """
    pca = PCA(n_components=2)
    X_3d = pca.fit_transform(X)

    fig = plt.figure(figsize= (10,10))
    ax = fig.add_subplot()

    ax.scatter(X_3d[:,0], X_3d[:,1] , c=labels, cmap='tab10', s=5, alpha=0.8)
    ax.legend(labels)
    plt.tight_layout()
    plt.show()



[5, 11, 20, 22, 29, 45, 46, 58, 107, 116, 117, 162, 202, 205, 226, 309, 369, 397, 424, 446, 506, 670, 671, 704, 705, 716, 728, 729, 730, 731, 734, 735, 801, 812, 813, 870, 876, 897, 899, 900, 902, 932, 959, 965, 978, 1208, 1244, 1271, 1272, 1291, 1292, 1293, 1314, 1315, 1328, 1420, 1430, 1506, 1510, 1512, 1514, 1515, 1520, 1570, 1571, 1572, 1573, 1574, 1577, 1580, 1584, 1585, 1586, 1587, 1588, 1613, 1641, 1726, 1738, 1778, 1804, 1806, 1814, 1864, 1952, 1982, 2016, 2091, 2119, 2121, 2122, 2125, 2142, 2150, 2173, 2176, 2389, 2401, 2418, 2435, 2610, 2657, 2669, 2674, 2680, 2754, 2757, 2801, 2807, 2826, 2828, 2834, 2847, 2872, 2873, 2874, 2909, 2920, 2965, 2980, 3035, 3047, 3081, 3093, 3129, 3133, 3158, 3229, 3266, 3338, 3345, 3504, 3510, 3530, 3576, 3597, 3605, 3615, 3669, 3673, 3764, 3818, 3890, 3891, 4047, 4070, 4132, 4138, 4168, 4221, 4237, 4247, 4254, 4256, 4257, 4259, 4295, 4320, 4321, 4327, 4331, 4362, 4363, 4369, 4382, 4383, 4384, 4385, 4386, 4387, 4388, 4389, 4390, 4419, 4507, 451